In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report


In [2]:
def load_data(path="../data/meter_data.csv"):
    df = pd.read_csv(path)
    df["feed_vs_declared_ratio"] = df["transformer_feed_estimate_kwh"] / df["declared_kwh"].clip(lower=1)
    return df


In [3]:
def train_model(df, contamination=0.08):
    feature_cols = [
        "declared_kwh", "transformer_feed_estimate_kwh", "historical_avg_kwh",
        "pct_drop_recent", "night_usage_ratio", "area_theft_history_rate",
        "feed_vs_declared_ratio"
    ]
    X = df[feature_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(contamination=contamination, random_state=42)
    model.fit(X_scaled)  # <-- train the model

    anomaly_scores = model.decision_function(X_scaled)
    flagged = model.predict(X_scaled)

    return model, scaler, feature_cols, anomaly_scores, flagged


In [4]:
def score_all_meters(df, anomaly_scores, flagged):
    result = df.copy()
    result["anomaly_score"] = -anomaly_scores
    result["flagged"] = (flagged == -1).astype(int)
    result["tier"] = pd.qcut(result["anomaly_score"], q=3, labels=["low", "medium", "high"])
    result.to_csv("../data/meter_anomaly_scores.csv", index=False)  
    return result.sort_values("anomaly_score", ascending=False)


In [5]:
df = load_data()
model, scaler, feature_cols, anomaly_scores, flagged = train_model(df)
scored = score_all_meters(df, anomaly_scores, flagged)

print("ROC-AUC:", roc_auc_score(df["is_theft"], scored["anomaly_score"]))
print("Average Precision:", average_precision_score(df["is_theft"], scored["anomaly_score"]))
print(classification_report(df["is_theft"], scored["flagged"]))


ROC-AUC: 0.5
Average Precision: 0.5
              precision    recall  f1-score   support

           0       0.50      0.67      0.57         3
           1       0.00      0.00      0.00         2

    accuracy                           0.40         5
   macro avg       0.25      0.33      0.29         5
weighted avg       0.30      0.40      0.34         5

